# Stoneforge RL — Training Notebook

Funktioniert **lokal (Jupyter)** und auf **Google Colab**.

| Schritt | Was passiert |
|---|---|
| 1. Konfiguration | GitHub-URL + Drive-Optionen setzen |
| 2. Setup | Repo klonen (Colab) oder lokal erkennen, Drive mounten |
| 3. Build | C++ Binding `stoneforge_sim.so` kompilieren |
| 4. Deps | Python-Pakete installieren |
| 5. Sanity | Env einmal testen (reset + step) |
| 6. Training | PPO mit `SeedEvalCallback` (50 Seeds, TensorBoard) |
| 7. Eval | 50-Seed-Test auf 7000–7049 → Ergebnistabelle |

**Rebuild nach C++-Änderungen:** Zelle 3 erneut ausführen.

## 1 — Konfiguration
Hier alle Optionen setzen, dann alle Zellen ausführen (`Runtime → Run all`).

In [ ]:
# ──────────────────────────────────────────────────────
# KONFIGURATION — hier anpassen
# ──────────────────────────────────────────────────────

# GitHub-URL des Repos (nur für Colab nötig; leer lassen wenn lokal)
GITHUB_URL = ""  # z.B. "https://github.com/dein-name/stoneforge-rl.git"

# Google Drive für persistente Model-Speicherung (nur Colab)
USE_GDRIVE = False
GDRIVE_SAVE_DIR = "/content/drive/MyDrive/stoneforge_models"

# Training
ALGO       = "ppo"        # "ppo" oder "dqn"
TIMESTEPS  = 1_000_000
N_ENVS     = 4            # auf Colab eher 4; lokal bis 8
EVAL_FREQ  = 50_000       # alle N Steps evaluieren

# Eval-Seeds (Standard: 7000–7049 = Testset A)
EVAL_SEEDS = list(range(7000, 7050))

## 2 — Umgebung erkennen & Repo vorbereiten

In [ ]:
import os, sys

# Colab-Erkennung
try:
    import google.colab
    IN_COLAB = True
    print("Umgebung: Google Colab")
except ImportError:
    IN_COLAB = False
    print("Umgebung: Lokal / Jupyter")

# ── Google Drive mounten (optional) ────────────────────
if IN_COLAB and USE_GDRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(GDRIVE_SAVE_DIR, exist_ok=True)
    print(f"Drive gemountet → Modelle werden in {GDRIVE_SAVE_DIR} gespeichert")

# ── Repo-Root bestimmen ────────────────────────────────
if IN_COLAB:
    REPO_ROOT = "/content/stoneforge"
    if not os.path.exists(REPO_ROOT):
        if GITHUB_URL:
            os.system(f"git clone {GITHUB_URL} {REPO_ROOT}")
        else:
            raise RuntimeError(
                "Colab erkannt, aber GITHUB_URL ist leer.\n"
                "Entweder GITHUB_URL setzen oder das Repo manuell nach "
                f"{REPO_ROOT} kopieren."
            )
    os.chdir(REPO_ROOT)
else:
    # Lokal: vom Notebook-Ordner eine Ebene hoch
    REPO_ROOT = os.path.dirname(os.path.abspath("."))
    # Falls das Notebook direkt im Repo-Root liegt:
    if os.path.exists(os.path.join(REPO_ROOT, "CMakeLists.txt")):
        pass
    elif os.path.exists("CMakeLists.txt"):
        REPO_ROOT = os.getcwd()
    else:
        REPO_ROOT = os.path.expanduser("~/Master_Projektarbeit")
    os.chdir(REPO_ROOT)

print(f"Repo-Root: {REPO_ROOT}")
assert os.path.exists("CMakeLists.txt"), f"CMakeLists.txt nicht gefunden in {os.getcwd()}"

## 3 — C++ Binding bauen

Baut nur `stoneforge_sim.so` (kein raylib, kein headless).  
Beim ersten Mal lädt CMake `nlohmann_json` + `pybind11` herunter (~30 Sek.  
Folge-Builds mit Cache sind schnell (<10 Sek.).

> **Lokale Nutzer:** Falls `.so` schon existiert, überspringen und direkt zu Zelle 4.

In [ ]:
import subprocess, shutil

BUILD_DIR = os.path.join(REPO_ROOT, "build")

# Auf Colab: cmake und g++ sind vorinstalliert
# Lokale macOS-User: brew install cmake falls nötig
def run(cmd, **kwargs):
    print(f"$ {cmd}")
    result = subprocess.run(cmd, shell=True, text=True,
                            capture_output=True, **kwargs)
    if result.stdout:
        print(result.stdout[-3000:])   # letzte 3000 Zeichen
    if result.returncode != 0:
        print("STDERR:", result.stderr[-2000:])
        raise RuntimeError(f"Befehl fehlgeschlagen: {cmd}")

cmake_configure = (
    f"cmake -S {REPO_ROOT} -B {BUILD_DIR}"
    f" -DCMAKE_BUILD_TYPE=Release"
    f" -DBUILD_PYTHON_BINDINGS=ON"
    f" -DBUILD_RAYLIB_CLIENT=OFF"
    f" -DBUILD_HEADLESS_RUNNER=OFF"
    f" -DBUILD_SDL_CLIENT=OFF"
)

ncpus = os.cpu_count() or 2
cmake_build = f"cmake --build {BUILD_DIR} --target stoneforge_sim -j{ncpus}"

run(cmake_configure)
run(cmake_build)

# .so in python/ kopieren (damit PYTHONPATH einfach bleibt)
so_src = os.path.join(BUILD_DIR, "stoneforge_sim.so")
so_dst = os.path.join(REPO_ROOT, "python", "stoneforge_sim.so")
if os.path.exists(so_src):
    shutil.copy2(so_src, so_dst)
    print(f"✓ stoneforge_sim.so → {so_dst}")
else:
    print(f"WARNUNG: {so_src} nicht gefunden — prüfe Build-Output oben")

## 4 — Python-Pakete installieren

In [ ]:
import subprocess
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "stable-baselines3>=2.3",
     "gymnasium>=0.29",
     "numpy>=1.26",
     "tensorboard>=2.14"],
    check=True
)
print("✓ Python-Pakete installiert")

## 5 — PYTHONPATH + Imports

In [ ]:
# build/ und python/ in Suchpfad aufnehmen
for p in [os.path.join(REPO_ROOT, "build"),
          os.path.join(REPO_ROOT, "python")]:
    if p not in sys.path:
        sys.path.insert(0, p)

os.chdir(REPO_ROOT)   # working dir = Repo-Root (für game_config.json)

import numpy as np
import stoneforge_sim          # C++ Binding
from stoneforge_env import StoneforgeWorldEnv

print(f"✓ stoneforge_sim geladen: {stoneforge_sim.__file__}")
print(f"  stoneforge_env: {StoneforgeWorldEnv}")

## 6 — Sanity Check

Prüft: Env startet, Observation hat richtige Form, Step liefert korrekten Reward.

In [ ]:
env = StoneforgeWorldEnv()
obs, info = env.reset(seed=42)

print(f"Observation shape : {obs.shape}   (erwartet: (455,))")
print(f"Obs min / max     : {obs.min():.3f} / {obs.max():.3f}   (soll in [-1, 1])")
print(f"Action space      : {env.action_space}")
print(f"BFS-Distanz Start : {info}")

# Ein Schritt in Richtung Exit (exitDx/exitDy an Positionen -2/-1)
dx = obs[-2]; dy = obs[-1]
action = 3 if abs(dx) >= abs(dy) else (1 if dy > 0 else 0)
obs2, reward, term, trunc, info2 = env.step(action)

print(f"\nNach 1 Schritt:")
print(f"  reward     = {reward:.4f}   (erwartet bei Fortschritt: ca. +0.025 bis +0.035)")
print(f"  terminated = {term}, truncated = {trunc}")
print(f"  BFS-Distanz: {info2.get('bfs_distance', '?')}")

assert obs.shape == (455,), f"Falsche Obs-Shape: {obs.shape}"
assert -1.1 < obs.min() and obs.max() < 1.1, "Obs außerhalb [-1, 1]"
print("\n✓ Sanity-Check bestanden")

## 6b — Reward-Diagnostik (vor Training ausführen!)

Prüft schrittweise ob PBRS korrekte Werte liefert.

**Was erwartet wird:**
- Schritt Richtung Exit: `reward ≈ +0.010` (netto nach Schrittstrafe)
- Schritt weg vom Exit: `reward ≈ −0.020`
- `mean reward/step ≈ −0.010 bis −0.005` bei zufälliger Policy

**Roter Flag:** `mean_reward_per_step > 0.05` oder `max_bfs_dist >> start_dist` → PBRS-Bug, rebuild nötig.

> Historische Ursache: BUFFER=20 in `computeBfsDistances()` → Agent wandert raus → Manhattan-Fallback springt von 35 auf 100+ → beim Rücklaufen: `β×(100−35)/128 = +2.54/Schritt` → falsches Lernziel. Fix: BUFFER=80 (bereits in simulation.cpp eingepflegt).

In [ ]:
import numpy as np

diag_env = StoneforgeWorldEnv()
obs, _ = diag_env.reset(seed=42)

init_bfs = diag_env.core.current_bfs_distance_to_exit()
print(f"BFS-Startdistanz : {init_bfs} Tiles")
print(f"exitDx (normalisiert): {obs[-2]:.3f}  →  {obs[-2]*64:.1f} Tiles")
print(f"exitDy (normalisiert): {obs[-1]:.3f}  →  {obs[-1]*64:.1f} Tiles")
print()

# 300 Schritte mit zufälliger Policy
rewards, bfs_dists = [], []
obs, _ = diag_env.reset(seed=42)
rng = np.random.default_rng(0)

for step in range(300):
    action = int(rng.integers(0, 4))
    obs, rew, term, trunc, info = diag_env.step(action)
    rewards.append(rew)
    bfs_dists.append(info.get("bfs_distance", -1))
    if term or trunc:
        obs, _ = diag_env.reset()

print("Zufällige Policy, 300 Steps:")
print(f"  mean reward/step : {np.mean(rewards):.4f}   (Ziel: ca. −0.010 bis −0.005)")
print(f"  min / max reward : {min(rewards):.3f} / {max(rewards):.3f}")
print(f"  mean BFS dist    : {np.mean(bfs_dists):.1f}   (Ziel: nahe {init_bfs})")
print(f"  max BFS dist     : {max(bfs_dists)}   (Warnung wenn >> {init_bfs + 15})")
print()

ok = True
if np.mean(rewards) > 0.05:
    print("⚠️  WARNUNG: mean reward > 0.05 bei zufälliger Policy!")
    print("   → Rebuild vergessen oder PBRS-Bug. Zelle 3 (Build) erneut ausführen.")
    ok = False

if max(bfs_dists) > init_bfs + 20:
    print(f"⚠️  Agent verlässt BFS-Box (max {max(bfs_dists)} >> start {init_bfs}).")
    print("   → BUFFER zu klein. Fix: BUFFER=80 in simulation.cpp + Rebuild.")
    ok = False

if ok:
    print("✓ Reward-Signal korrekt — Training kann starten.")

## 7 — Training

> **Empfehlung für den ersten Run: `ALGO = "dqn"`**  
> DQN hat bei identischem Setup historisch 40% erreicht, PPO bei spärlichem Reward deutlich weniger.  
> ε-greedy Exploration ist hier effektiver als PPO-Entropie-Bonus.  
> PPO danach als Vergleichs-Baseline.

`gamma=0.999` **muss** mit `PBRS_GAMMA=0.999` in `simulation.cpp` übereinstimmen — sonst ist PBRS nicht mehr policy-invariant.  
Der `SeedEvalCallback` evaluiert alle `EVAL_FREQ` Steps auf Seeds 7000–7049 und speichert das beste Modell.

**Laufzeit bei 1M Steps:** ca. 10–20 Min (CPU, 4 Envs).

In [ ]:
from stable_baselines3 import PPO, DQN
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.callbacks import BaseCallback

MAX_EVAL_STEPS = 4000


class SeedEvalCallback(BaseCallback):
    """Evaluiert alle eval_freq Rollout-Steps auf festen Seeds."""

    def __init__(self, seeds, eval_freq, save_dir, verbose=1):
        super().__init__(verbose)
        self.seeds    = seeds
        self.eval_freq = eval_freq
        self.save_dir  = save_dir
        self._best    = -1.0
        self.history  = []          # [(timestep, success_rate)]
        os.makedirs(save_dir, exist_ok=True)

    def _on_step(self):
        if self.n_calls % self.eval_freq == 0:
            self._eval()
        return True

    def _eval(self):
        env  = StoneforgeWorldEnv()
        succ = 0
        for seed in self.seeds:
            obs, _ = env.reset(seed=seed)
            done, steps = False, 0
            while not done and steps < MAX_EVAL_STEPS:
                action, _ = self.model.predict(obs, deterministic=True)
                obs, _, term, trunc, info = env.step(int(action))
                done = term or trunc; steps += 1
                if info.get("reached_exit"):
                    succ += 1; break

        rate = succ / len(self.seeds)
        self.history.append((self.num_timesteps, rate))
        self.logger.record("eval/success_rate", rate)
        self.logger.dump(self.num_timesteps)

        tag = "★ BEST " if rate > self._best else ""
        print(f"  {tag}[Eval @ {self.num_timesteps:>8,}] {succ}/{len(self.seeds)} = {rate:.1%}")

        if rate > self._best:
            self._best = rate
            self.model.save(os.path.join(self.save_dir, "best_model"))

    @property
    def best_rate(self):
        return self._best

In [ ]:
# Modell-Speicherort (Drive wenn aktiviert, sonst lokal)
if IN_COLAB and USE_GDRIVE:
    SAVE_DIR = os.path.join(GDRIVE_SAVE_DIR, f"best_models_{ALGO}")
else:
    SAVE_DIR = os.path.join(REPO_ROOT, f"best_models_{ALGO}")

print(f"Modelle werden gespeichert in: {SAVE_DIR}")

# Trainings-Envs
train_env = make_vec_env(StoneforgeWorldEnv, n_envs=N_ENVS)

eval_cb = SeedEvalCallback(
    seeds     = EVAL_SEEDS,
    eval_freq = max(1, EVAL_FREQ // N_ENVS),  # in Rollout-Steps
    save_dir  = SAVE_DIR,
)

PPO_KWARGS = dict(
    policy       = "MlpPolicy",
    n_steps      = 2048,
    batch_size   = 256,
    n_epochs     = 10,
    learning_rate= 3e-4,
    gamma        = 0.999,   # MUSS mit PBRS_GAMMA in simulation.cpp übereinstimmen!
    gae_lambda   = 0.95,
    clip_range   = 0.2,
    ent_coef     = 0.01,
    verbose      = 1,
    tensorboard_log = os.path.join(REPO_ROOT, "tensorboard_logs"),
)

DQN_KWARGS = dict(
    policy              = "MlpPolicy",
    learning_rate       = 1e-4,
    buffer_size         = 200_000,
    learning_starts     = 10_000,
    batch_size          = 256,
    gamma               = 0.999,
    exploration_fraction= 0.5,
    exploration_final_eps=0.05,
    train_freq          = 4,
    target_update_interval=1000,
    verbose             = 1,
    tensorboard_log     = os.path.join(REPO_ROOT, "tensorboard_logs"),
)

if ALGO == "ppo":
    model = PPO(env=train_env, **PPO_KWARGS)
else:
    model = DQN(env=train_env, **DQN_KWARGS)

print(f"\nStarte {ALGO.upper()} Training ({TIMESTEPS:,} Steps, {N_ENVS} Envs)...")
print(f"Eval alle {EVAL_FREQ:,} Steps auf {len(EVAL_SEEDS)} Seeds.")
print("-" * 60)

model.learn(
    total_timesteps   = TIMESTEPS,
    callback          = eval_cb,
    tb_log_name       = f"{ALGO}_run",
    reset_num_timesteps= True,
)

model.save(os.path.join(SAVE_DIR, "final_model"))
print(f"\n✓ Training beendet. Beste Success Rate: {eval_cb.best_rate:.1%}")
print(f"  Bestes Modell: {SAVE_DIR}/best_model.zip")

## 8 — TensorBoard

Zeigt `eval/success_rate` und alle SB3-Metriken live.

In [ ]:
tb_log_dir = os.path.join(REPO_ROOT, "tensorboard_logs")
%load_ext tensorboard
%tensorboard --logdir {tb_log_dir}

## 9 — Evaluierung (50-Seed-Test)

Lädt das beste Modell und evaluiert deterministisch auf Seeds 7000–7049.  
Entspricht dem Standardtest aus `Projektarbeit_RL_Dokumentation.md`.

In [ ]:
from stable_baselines3 import PPO, DQN
import numpy as np
from datetime import date

MODEL_PATH = os.path.join(SAVE_DIR, "best_model.zip")

if ALGO == "ppo":
    loaded_model = PPO.load(MODEL_PATH)
else:
    loaded_model = DQN.load(MODEL_PATH)

print(f"Modell geladen: {MODEL_PATH}")
print(f"Evaluiere auf Seeds {EVAL_SEEDS[0]}–{EVAL_SEEDS[-1]} ({len(EVAL_SEEDS)} Seeds)...")
print("-" * 60)

eval_env = StoneforgeWorldEnv()
successes, ep_lens, ep_returns = 0, [], []

for seed in EVAL_SEEDS:
    obs, _ = eval_env.reset(seed=seed)
    done, ep_ret, steps, reached = False, 0.0, 0, False
    while not done and steps < MAX_EVAL_STEPS:
        action, _ = loaded_model.predict(obs, deterministic=True)
        obs, r, term, trunc, info = eval_env.step(int(action))
        ep_ret += float(r); steps += 1
        if info.get("reached_exit"):
            reached = True
        done = term or trunc
    successes += int(reached)
    ep_lens.append(steps)
    ep_returns.append(ep_ret)

n = len(EVAL_SEEDS)
rate = successes / n
today = date.today().strftime("%d.%m.%Y")

print(f"\n{'='*60}")
print(f"  Ergebnis  {ALGO.upper()} — {today}")
print(f"{'='*60}")
print(f"  Erfolge          : {successes} / {n}")
print(f"  Success Rate     : {rate:.1%}")
print(f"  Mittl. Ep-Länge  : {np.mean(ep_lens):.1f}")
print(f"  Mittl. Return    : {np.mean(ep_returns):.2f}")
print(f"{'='*60}")

# Ziel: ≥ 70% auf Testset A
target = 0.70
status = "✓ ZIEL ERREICHT" if rate >= target else f"✗ Ziel {target:.0%} noch nicht erreicht"
print(f"  Projektziel (≥ 70%): {status}")

## 10 — Lernkurve visualisieren

In [ ]:
import matplotlib
import matplotlib.pyplot as plt

if eval_cb.history:
    steps_hist, rates_hist = zip(*eval_cb.history)
    plt.figure(figsize=(10, 4))
    plt.plot(steps_hist, [r * 100 for r in rates_hist], marker="o", linewidth=2)
    plt.axhline(70, color="green", linestyle="--", label="Ziel 70%")
    plt.xlabel("Timesteps")
    plt.ylabel("Success Rate (%)")
    plt.title(f"{ALGO.upper()} — Lernkurve (Seeds 7000–7049)")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(REPO_ROOT, f"lernkurve_{ALGO}.png"), dpi=120)
    plt.show()
else:
    print("Keine Eval-History verfügbar (Training wurde eventuell übersprungen).")

## 11 — Changelog-Eintrag generieren

Kopiere den Output in `Projektarbeit_RL_Dokumentation.md` (Ergebnistabelle in v2.0).

In [ ]:
print("Markdown für Projektarbeit_RL_Dokumentation.md:")
print()
print("| Algorithmus | Erfolge | Success Rate | Mittl. Episodenlänge | Mittl. Return | Datum |")
print("|---|---|---|---|---|---|")
print(
    f"| {ALGO.upper()} (β=5.0, γ=0.999) "
    f"| {successes} / {n} "
    f"| {rate:.1%} "
    f"| {np.mean(ep_lens):.1f} "
    f"| {np.mean(ep_returns):.2f} "
    f"| {today} |"
)